In [1]:
from model.model import get_model
import os
from dotenv import load_dotenv
from outlines import Generator, from_transformers  
from schema.ticket import Ticket 
from prompt.summarizer import summary_prompt
from schema.table import Table
import json
from evalution.metric import TicketEvaluator
load_dotenv()

True

In [2]:
location = os.getenv('DATA_FILE_NAME')
table_1 = Table(location)

In [3]:
name = os.getenv('MODEL_NAME')
hf_model,hf_tokenizer = get_model(name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


model downloaded


In [4]:
hf_model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [5]:
model = from_transformers(hf_model,hf_tokenizer)

In [6]:
generator = Generator(model,Ticket)

In [7]:
evaluator = TicketEvaluator(generator=generator)



In [ ]:
metric = evaluator.evalute_ticket(table_1.return_ticket(1),max_new_tokens=150,use_cache= True)

d:\Programing\Depi\ticket_summary\ticket_summary\.venv\Lib\site-packages\transformers\generation\utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=120) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
W0909 18:03:23.693000 36708 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] WON'T CONVERT _apply_token_bitmask_inplace_kernel d:\Programing\Depi\ticket_summary\ticket_summary\.venv\Lib\site-packages\outlines_core\kernels\torch.py line 43 
W0909 18:03:23.693000 36708 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] due to: 
W0909 18:03:23.693000 36708 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] Traceback (most recent call last):
W0909 18:03:23.693000 36708 Lib\site-packages\torch\_dynamo\convert_frame.py:2415]   File "d:\Programing\Depi\ticket_summary\ticket_summary\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2319, in __call__
W0909 18

AttributeError: 'numpy.int64' object has no attribute 'values'

In [ ]:
print(metric)

Role: You are a precise ticket classification assistant.

   task: Classify the customer ticket.

   Constraints: - Return only the requested result.
- Do not provide explanations.
- Do not use markdown.
- summary must be one sentence as max

    input : I forgot my password and need to reset it.


In [ ]:
result = generator(summary_prompt(table_1.return_ticket(2)), max_new_tokens=150,)
#print(result)

In [ ]:
output = Ticket.model_validate_json(result)
print(output.model_dump_json())

{"category":"delivery","sentiment":"negative","urgency":"medium","summary":"Package delivery was delayed for two days."}


In [ ]:
row = table_1.return_row(1)

In [ ]:
row['urgency'].values

<ArrowStringArray>
['high']
Length: 1, dtype: str

In [ ]:
table_1.return_row(1)

,ticket_id,message,category,sentiment,urgency
8,9,The mobile app freezes when I upload a file.,technical,negative,high


In [ ]:
table_1.return_samples(1)

8    The mobile app freezes when I upload a file.
Name: message, dtype: str